# Day 02: Calculus & Gradients — How Models Learn

**Goal:** Understand derivatives, gradients, and gradient descent — the algorithm that trains every LLM.

Yesterday: `output = input @ weights + bias` (one layer).  
Today: **how does the model figure out the right weights?**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. What is a Derivative? (Slope = Rate of Change)

The derivative tells you: **"if I nudge x a tiny bit, how much does f(x) change?"**

- Positive derivative → function is going UP
- Negative derivative → function is going DOWN  
- Zero derivative → function is FLAT (could be a minimum!)

In [ ]:
# Let's visualize f(x) = x² and its derivative f'(x) = 2x

def f(x):
    return x ** 2

def f_derivative(x):
    return 2 * x

x = np.linspace(-4, 4, 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Plot the function
ax1.plot(x, f(x), 'b-', linewidth=2)
ax1.set_title("f(x) = x²")
ax1.set_xlabel("x")
ax1.set_ylabel("f(x)")
ax1.axhline(y=0, color='k', linewidth=0.5)
ax1.axvline(x=0, color='k', linewidth=0.5)
ax1.grid(True, alpha=0.3)

# Mark some points and their slopes
for xi, color in [(-3, 'red'), (0, 'green'), (2, 'orange')]:
    ax1.plot(xi, f(xi), 'o', color=color, markersize=10)
    # Draw tangent line
    slope = f_derivative(xi)
    tangent_x = np.linspace(xi - 1.5, xi + 1.5, 50)
    tangent_y = f(xi) + slope * (tangent_x - xi)
    ax1.plot(tangent_x, tangent_y, '--', color=color, alpha=0.7, 
             label=f"x={xi}, slope={slope}")

ax1.legend()

# Plot the derivative
ax2.plot(x, f_derivative(x), 'r-', linewidth=2)
ax2.set_title("f'(x) = 2x (the derivative)")
ax2.set_xlabel("x")
ax2.set_ylabel("f'(x) = slope")
ax2.axhline(y=0, color='k', linewidth=0.5)
ax2.axvline(x=0, color='k', linewidth=0.5)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("At x=-3: slope = -6 (going DOWN → move right to reduce f)")
print("At x= 0: slope =  0 (FLAT → this is the minimum!)")
print("At x= 2: slope =  4 (going UP → move left to reduce f)")

## 2. Numerical Derivative — Computing Slope Without Calculus

You don't need to know calculus rules. Just nudge x by a tiny amount `h` and see how f changes.

```
f'(x) ≈ (f(x + h) - f(x - h)) / (2 * h)
```

This is how you can verify any derivative — and it's how gradient checking works in practice.

In [ ]:
def numerical_derivative(f, x, h=1e-5):
    """Compute derivative of f at x using tiny nudge."""
    return (f(x + h) - f(x - h)) / (2 * h)

# Test on f(x) = x²  →  f'(x) should be 2x
def f(x):
    return x ** 2

# Compare numerical vs analytical at several points
print("x   | Numerical  | Analytical (2x) | Match?")
print("-" * 50)
for x_val in [-3, -1, 0, 1, 3]:
    num = numerical_derivative(f, x_val)
    ana = 2 * x_val
    print(f"{x_val:3d} | {num:10.6f} | {ana:15.6f} | {'✓' if abs(num - ana) < 1e-4 else '✗'}")

# Works for ANY function — even ones where calculus is hard
def mystery(x):
    return np.sin(x) * x ** 2 + np.exp(-x)

print(f"\nmystery function derivative at x=1: {numerical_derivative(mystery, 1.0):.6f}")
print("(No need to do the calculus by hand!)")

## 3. Gradient Descent — Finding the Minimum

This is THE algorithm that trains every neural network. The idea:

1. Start at a random point
2. Compute the slope (derivative)
3. Take a step opposite to the slope
4. Repeat until you reach the bottom

`x_new = x_old - learning_rate × derivative`

In [ ]:
# Gradient descent on f(x) = x²
# The minimum is at x=0, but let's pretend we don't know that

def f(x):
    return x ** 2

def f_deriv(x):
    return 2 * x

# Start at a random point
x = 4.0
learning_rate = 0.1
history = [(x, f(x))]

print(f"Step  0: x = {x:.4f}, f(x) = {f(x):.4f}")

for step in range(1, 21):
    gradient = f_deriv(x)
    x = x - learning_rate * gradient  # THE update rule
    history.append((x, f(x)))
    if step <= 10 or step % 5 == 0:
        print(f"Step {step:2d}: x = {x:.4f}, f(x) = {f(x):.4f}, gradient = {gradient:.4f}")

print(f"\nFinal: x ≈ {x:.6f} (should be ~0.0)")
print(f"       f(x) ≈ {f(x):.10f} (should be ~0.0)")

In [ ]:
# Visualize the gradient descent path
x_range = np.linspace(-5, 5, 100)
hist_x = [h[0] for h in history]
hist_y = [h[1] for h in history]

plt.figure(figsize=(10, 5))
plt.plot(x_range, f(x_range), 'b-', linewidth=2, label='f(x) = x²')
plt.plot(hist_x, hist_y, 'ro-', markersize=6, label='Gradient descent steps')
plt.plot(hist_x[0], hist_y[0], 'gs', markersize=12, label=f'Start (x={hist_x[0]})')
plt.plot(hist_x[-1], hist_y[-1], 'r*', markersize=15, label=f'End (x={hist_x[-1]:.3f})')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('Gradient Descent: Rolling Downhill to the Minimum')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Learning Rate — Too High, Too Low, Just Right

The learning rate controls step size. It's the most important hyperparameter in training.

In [ ]:
# Compare different learning rates
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x_range = np.linspace(-5, 5, 100)

for ax, lr, title in zip(axes, [0.01, 0.1, 0.9],
                           ["Too Small (0.01)\nSlow convergence",
                            "Just Right (0.1)\nSmooth convergence", 
                            "Too Large (0.9)\nOscillates wildly"]):
    x = 4.0
    path_x, path_y = [x], [f(x)]
    
    for _ in range(20):
        x = x - lr * f_deriv(x)
        path_x.append(x)
        path_y.append(f(x))
    
    ax.plot(x_range, f(x_range), 'b-', linewidth=2)
    ax.plot(path_x, path_y, 'ro-', markersize=4)
    ax.set_title(title)
    ax.set_ylim(-1, 20)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Partial Derivatives & Gradients (Multiple Variables)

In a real model, you have millions of weights. The gradient is a vector of partial derivatives — one per weight. Let's start with 2 variables to visualize it.

In [ ]:
# f(x, y) = x² + y²  →  minimum at (0, 0)
# ∂f/∂x = 2x,  ∂f/∂y = 2y
# gradient = [2x, 2y]

def f_2d(x, y):
    return x**2 + y**2

def gradient_2d(x, y):
    return np.array([2*x, 2*y])

# Gradient descent in 2D
pos = np.array([4.0, 3.0])  # start position
lr = 0.1
path = [pos.copy()]

for step in range(30):
    grad = gradient_2d(pos[0], pos[1])
    pos = pos - lr * grad
    path.append(pos.copy())

path = np.array(path)

# Visualize with contour plot
fig, ax = plt.subplots(figsize=(8, 8))
x_grid = np.linspace(-5, 5, 100)
y_grid = np.linspace(-5, 5, 100)
X, Y = np.meshgrid(x_grid, y_grid)
Z = f_2d(X, Y)

ax.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.6)
ax.plot(path[:, 0], path[:, 1], 'ro-', markersize=4, label='Gradient descent path')
ax.plot(path[0, 0], path[0, 1], 'gs', markersize=12, label='Start')
ax.plot(path[-1, 0], path[-1, 1], 'r*', markersize=15, label='End')
ax.set_xlabel('x (weight 1)')
ax.set_ylabel('y (weight 2)')
ax.set_title('2D Gradient Descent: Rolling Down a Bowl')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.show()

print(f"Started at: ({path[0, 0]:.1f}, {path[0, 1]:.1f})")
print(f"Ended at:   ({path[-1, 0]:.4f}, {path[-1, 1]:.4f})")
print(f"Minimum is at (0, 0)")

## 6. Chain Rule — The Heart of Backpropagation

A neural network is a chain of functions: `layer3(layer2(layer1(input)))`.

The chain rule says: to find how the final output changes when you tweak an early weight, **multiply the derivatives through each layer**.

```
y = f(g(h(x)))
dy/dx = f'(g(h(x))) × g'(h(x)) × h'(x)
```

This backward multiplication is called **backpropagation**.

In [ ]:
# Chain rule example: y = (3x + 2)²
# Break it down:  h(x) = 3x + 2,  f(h) = h²
# dy/dx = f'(h) × h'(x) = 2h × 3 = 2(3x+2) × 3 = 6(3x+2)

def y(x):
    return (3 * x + 2) ** 2

def dy_dx_analytical(x):
    return 6 * (3 * x + 2)

# Verify with numerical derivative
x_test = 1.0
num = numerical_derivative(y, x_test)
ana = dy_dx_analytical(x_test)
print(f"y = (3x + 2)² at x = {x_test}")
print(f"Numerical derivative:  {num:.6f}")
print(f"Analytical (chain rule): {ana:.6f}")
print(f"Match: {'✓' if abs(num - ana) < 1e-4 else '✗'}")

# Now a 3-layer chain: y = sin(exp(x²))
# h1(x) = x²     →  h1' = 2x
# h2(z) = exp(z)  →  h2' = exp(z)
# h3(z) = sin(z)  →  h3' = cos(z)
# dy/dx = cos(exp(x²)) × exp(x²) × 2x

def y_chain(x):
    return np.sin(np.exp(x ** 2))

def dy_chain_analytical(x):
    return np.cos(np.exp(x**2)) * np.exp(x**2) * 2 * x

x_test = 0.5
num = numerical_derivative(y_chain, x_test)
ana = dy_chain_analytical(x_test)
print(f"\ny = sin(exp(x²)) at x = {x_test}")
print(f"Numerical:  {num:.6f}")
print(f"Chain rule: {ana:.6f}")
print(f"Match: {'✓' if abs(num - ana) < 1e-4 else '✗'}")
print("\nThis is EXACTLY what backpropagation does — chain rule through every layer!")

## 7. Mini Project: Train a Line to Fit Data (Linear Regression from Scratch)

Let's put it ALL together. We have data points, and we want to find `y = wx + b` that fits them best. We'll use gradient descent to learn `w` (weight) and `b` (bias) — exactly like a single neuron.

In [ ]:
# Generate some noisy data: y = 3x + 7 (true relationship)
np.random.seed(42)
X = np.random.uniform(-5, 5, 50)
y_true = 3 * X + 7 + np.random.randn(50) * 2  # add noise

# Initialize random weight and bias
w = np.random.randn()  # should learn → 3
b = np.random.randn()  # should learn → 7
learning_rate = 0.01

losses = []

print(f"Starting: w={w:.3f} (target: 3.0), b={b:.3f} (target: 7.0)\n")

for epoch in range(100):
    # Forward pass: predictions
    y_pred = w * X + b
    
    # Loss: Mean Squared Error = average of (prediction - actual)²
    loss = np.mean((y_pred - y_true) ** 2)
    losses.append(loss)
    
    # Gradients (partial derivatives of loss w.r.t. w and b)
    # d(loss)/dw = mean(2 * (y_pred - y_true) * X)
    # d(loss)/db = mean(2 * (y_pred - y_true))
    dw = np.mean(2 * (y_pred - y_true) * X)
    db = np.mean(2 * (y_pred - y_true))
    
    # Update weights
    w = w - learning_rate * dw
    b = b - learning_rate * db
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}: loss={loss:.3f}, w={w:.3f}, b={b:.3f}")

print(f"\nFinal:  w={w:.3f} (target: 3.0), b={b:.3f} (target: 7.0)")
print(f"The model LEARNED the relationship from data!")

In [ ]:
# Visualize: the learned line + loss curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: data + learned line
ax1.scatter(X, y_true, alpha=0.6, label='Data points')
x_line = np.linspace(-5, 5, 100)
ax1.plot(x_line, w * x_line + b, 'r-', linewidth=2, 
         label=f'Learned: y = {w:.2f}x + {b:.2f}')
ax1.plot(x_line, 3 * x_line + 7, 'g--', linewidth=2, 
         label='True: y = 3x + 7', alpha=0.5)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_title('Learned Line vs True Line')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: loss over time
ax2.plot(losses, 'b-', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss (MSE)')
ax2.set_title('Loss Curve — Model Getting Better Over Time')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The loss curve going down = the model is learning!")
print("This same pattern applies to training GPT — just with billions of weights.")

## Exercises

1. **Learning rate experiment:** Re-run the linear regression with `learning_rate = 0.001` and `learning_rate = 0.5`. Plot the loss curves — what happens?

2. **More variables:** Modify the mini project to learn `y = 2x₁ + 5x₂ - 3`. You'll need 2 weights and 1 bias. (Hint: generate 2D input data)

3. **Numerical gradient check:** For the linear regression, compute dw and db numerically (nudge w by h, see how loss changes). Verify they match the analytical gradients.

4. **Non-linear function:** Try gradient descent on `f(x) = x⁴ - 3x³ + 2`. Does it find the global minimum? (Hint: it might get stuck in a local minimum — this is a real problem in ML!)

---

## Key Takeaways

- **Derivative** = slope = "which way is downhill?"
- **Gradient** = vector of all partial derivatives (one per weight)
- **Gradient descent** = take small steps opposite to the gradient → minimize loss
- **Learning rate** = step size. Too big → chaos. Too small → slow. Just right → convergence
- **Chain rule** = multiply derivatives through each layer → backpropagation
- **Loss curve going down** = the model is learning
- Tomorrow: PyTorch does ALL of this automatically with **autograd**!